# DECOMMISIONED USE THE SCRIPT INSTEAD

This is just to get a better idea of Adelaide's DZNs - how big they are, where they are, etc. so I can get a better idea of how to use them in modelling.

In [1]:
import osmnx as ox
import matplotlib.pyplot as plt
import geopandas as gpd

ox.settings.log_console = True
ox.settings.use_cache = True

In [2]:
gdf_boundary = ox.geocode_to_gdf("R11381689", by_osmid=True)
polygon = gdf_boundary.geometry.iloc[0]

G = ox.graph_from_polygon(polygon, network_type="bike")
print(f"Nodes: {len(G.nodes)}. Edges: {len(G.edges)}")
print(gdf_boundary.display_name[0])

Nodes: 125096. Edges: 297491
Adelaide, South Australia, Australia


In [3]:
dzns_aus = gpd.read_file("../all-data/australia-destination-zones-raw/DZN_2021_AUST_GDA2020.shp")
dzn_gdf_boundary_crs = gdf_boundary.to_crs(dzns_aus.crs)
dzns = gpd.clip(dzns_aus, dzn_gdf_boundary_crs)
print('DZN all count: ', len(dzns_aus))
print('DZN Adelaide count: ', len(dzns))

DZN all count:  9329
DZN Adelaide count:  479


In [4]:
import networkx as nx
nodes_gdf = ox.graph_to_gdfs(G, nodes=True, edges=False)[["geometry"]]

nodes_dzns = gpd.sjoin(
    nodes_gdf.to_crs(dzns.crs),
    dzns[["DZN_CODE21", "geometry"]],
    how="left",
    predicate="within",
)
node_dzn_codes = nodes_dzns["DZN_CODE21"].dropna().to_dict()
nx.set_node_attributes(G, node_dzn_codes, "DZN_CODE21")
print(f"Nodes with DZN_CODE21: {len(node_dzn_codes)} / {len(G.nodes)}")

Nodes with DZN_CODE21: 125077 / 125096


In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
import numpy as np

# Read the POW population data
pow_df = pd.read_csv("../all-data/census-formatted/greater-adelaide/EETP-POW-DZN.csv", usecols=["DZN (POW)", "Total"])
pow_df.columns = ["DZN_CODE21", "POW_Population"]
pow_df["DZN_CODE21"] = pow_df["DZN_CODE21"].astype(str)

# Create a mapping from DZN_CODE21 to POW Population
pow_mapping = pow_df.set_index("DZN_CODE21")["POW_Population"].to_dict()

# Get DZN_CODE21 for each node and map POW population
node_dzn = nx.get_node_attributes(G, "DZN_CODE21")
node_pow_pop = {}
for node, dzn in node_dzn.items():
    dzn_str = str(dzn)
    if dzn_str in pow_mapping:
        node_pow_pop[node] = pow_mapping[dzn_str]
    else:
        node_pow_pop[node] = 0

nx.set_node_attributes(G, node_pow_pop, "POW_Population")

print(f"Nodes with POW Population data: {sum(1 for v in node_pow_pop.values() if v > 0)} / {len(G.nodes)}")
print(f"POW Population range: {min(node_pow_pop.values())} - {max(node_pow_pop.values())}")

Nodes with POW Population data: 125075 / 125096
POW Population range: 0 - 13984


In [ ]:
# Plot the graph with POW population
fig, ax = plt.subplots(figsize=(30, 30), dpi=300)

# Get node positions (x, y coordinates)
pos = {node: (data['x'], data['y']) for node, data in G.nodes(data=True) if 'x' in data and 'y' in data}

# Get POW population for each node
node_populations = [G.nodes[node].get('POW_Population', 0) for node in pos.keys()]

# Create color map - redder for higher population
# Use log scale for better visualization since populations vary widely
max_pop = max(node_populations) if node_populations else 1
min_pop = min([p for p in node_populations if p > 0]) if any(p > 0 for p in node_populations) else 1

# Normalize using log scale for better color distribution
norm = mcolors.LogNorm(vmin=min_pop, vmax=max_pop)
cmap = plt.cm.Reds
node_colors = [cmap(norm(p)) if p > 0 else (0.9, 0.9, 0.9, 0.3) for p in node_populations]

# Node sizes based on population (larger for higher population)
node_sizes = [max(5, np.log1p(p) * 10) for p in node_populations]

# Draw edges first (light gray, thin)
nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#cccccc', width=0.2, alpha=0.5)

# Draw nodes
scatter = nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=node_sizes, alpha=0.8)

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.5, aspect=20)
cbar.set_label('POW Population (log scale)', fontsize=14)

# Add population labels for nodes with significant population
# Only label nodes with population > threshold to avoid clutter
label_threshold = 1000  # Only label nodes with POW population > 1000
for node, (x, y) in pos.items():
    pop = G.nodes[node].get('POW_Population', 0)
    if pop > label_threshold:
        ax.text(x, y, f'{int(pop):,}', fontsize=6, ha='center', va='center',
                fontweight='bold', color='darkblue',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor='none'))

ax.set_title('Greater Adelaide DZNs - POW Population per Destination Zone', fontsize=20, fontweight='bold')
ax.axis('off')
plt.tight_layout()

# Save as high-resolution PNG for zooming
output_path = "dzn_pow_population_graph.png"
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Graph saved to {output_path}")

plt.show()

KeyboardInterrupt: 